# 🛡️ StabilityGuard Quickstart Guide

This notebook demonstrates how to use StabilityGuard to protect your PyTorch training runs from gradient spikes and NaN explosions.

**What you'll learn:**
- Installation and setup
- Basic usage with a simple model
- Spike injection demo
- Visualization of gradient monitoring
- All three remediation strategies (skip, rollback, raise)

## 📦 Installation

Install StabilityGuard from PyPI:

In [ ]:
# Uncomment to install
# !pip install stabilityguard

# Or install from source for development
# !pip install -e .

## 🔧 Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict

from stabilityguard import GuardedOptimizer

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 🧠 Define a Simple Model

Let's create a small neural network for demonstration:

In [ ]:
class SimpleNet(nn.Module):
    """A simple 3-layer neural network for demonstration."""
    
    def __init__(self, input_dim=10, hidden_dim=64, output_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Create model and move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNet().to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nModel architecture:\n{model}")

## 🎯 Basic Usage: Wrap Your Optimizer

StabilityGuard is a drop-in replacement - just wrap your existing optimizer!

In [ ]:
# Create a standard PyTorch optimizer
base_optimizer = optim.Adam(model.parameters(), lr=0.001)

# Wrap it with StabilityGuard (default: skip action)
optimizer = GuardedOptimizer(
    optimizer=base_optimizer,
    model=model,
    action='skip',           # Skip the update if spike detected
    threshold=10.0,          # Spike threshold (10x baseline)
    warmup_steps=5,          # Don't detect spikes in first 5 steps
    ema_alpha=0.01,          # EMA smoothing factor
    log_dir='./sg_logs'      # Where to save logs
)

print("✅ Optimizer wrapped with StabilityGuard!")
print(f"   Action: {optimizer.action}")
print(f"   Threshold: {optimizer.spike_detector.threshold}x baseline")
print(f"   Warmup: {optimizer.spike_detector.warmup_steps} steps")

## 📊 Generate Synthetic Data

In [ ]:
def generate_data(n_samples=100, input_dim=10):
    """Generate synthetic classification data."""
    X = torch.randn(n_samples, input_dim)
    y = (X.sum(dim=1) > 0).long()  # Simple binary classification
    return X, y

# Generate training data
X_train, y_train = generate_data(n_samples=200)
X_train, y_train = X_train.to(device), y_train.to(device)

print(f"Training data shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")
print(f"Class distribution: {torch.bincount(y_train)}")

## 🏃 Normal Training (No Spikes)

First, let's train normally to see baseline behavior:

In [ ]:
def train_step(model, optimizer, X, y, criterion):
    """Single training step."""
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(X)
    loss = criterion(outputs, y)
    
    # Backward pass
    loss.backward()
    
    # Optimizer step (StabilityGuard monitors here)
    optimizer.step()
    
    return loss.item()

# Training loop
criterion = nn.CrossEntropyLoss()
losses = []

print("Training for 20 steps (no spike injection)...\n")

for step in range(20):
    loss = train_step(model, optimizer, X_train, y_train, criterion)
    losses.append(loss)
    
    if step % 5 == 0:
        print(f"Step {step:2d} | Loss: {loss:.4f}")

print("\n✅ Normal training completed - no spikes detected!")

## 💥 Spike Injection Demo

Now let's inject an artificial gradient spike to see StabilityGuard in action:

In [ ]:
def inject_spike(model, layer_name='fc2.weight', magnitude=1000.0):
    """Inject an artificial gradient spike into a specific layer."""
    for name, param in model.named_parameters():
        if name == layer_name and param.grad is not None:
            param.grad.data += magnitude * torch.randn_like(param.grad)
            print(f"💥 Injected spike into {name} (magnitude: {magnitude})")
            return True
    return False

# Reset model and optimizer for clean demo
model = SimpleNet().to(device)
base_optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = GuardedOptimizer(
    optimizer=base_optimizer,
    model=model,
    action='skip',
    threshold=10.0,
    warmup_steps=5
)

spike_losses = []
spike_detected_at = []

print("Training with spike injection at step 10...\n")

for step in range(20):
    # Normal training step
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    
    # Inject spike at step 10
    if step == 10:
        inject_spike(model, magnitude=1000.0)
    
    # StabilityGuard will detect and handle the spike
    optimizer.step()
    
    spike_losses.append(loss.item())
    
    # Check if spike was detected
    if hasattr(optimizer, '_last_snapshot') and optimizer._last_snapshot:
        if optimizer._last_snapshot.spike_detected:
            spike_detected_at.append(step)
    
    if step % 5 == 0 or step == 10:
        status = "🛡️ SPIKE DETECTED & SKIPPED" if step in spike_detected_at else "✓"
        print(f"Step {step:2d} | Loss: {loss:.4f} | {status}")

print(f"\n✅ Training completed! Spikes detected at steps: {spike_detected_at}")

## 📈 Visualize Gradient Monitoring

Let's visualize how StabilityGuard tracks gradient norms over time:

In [ ]:
# Collect gradient statistics during training
model = SimpleNet().to(device)
base_optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = GuardedOptimizer(
    optimizer=base_optimizer,
    model=model,
    action='skip',
    threshold=10.0,
    warmup_steps=5
)

# Track metrics
grad_norms_history = {name: [] for name, _ in model.named_parameters()}
ema_baselines_history = {name: [] for name, _ in model.named_parameters()}
spike_flags = []

print("Collecting gradient statistics...\n")

for step in range(30):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    
    # Inject spike at step 15
    if step == 15:
        inject_spike(model, magnitude=500.0)
    
    # Capture gradient norms before step
    snapshot = optimizer._capture_gradient_snapshot()
    
    for name in grad_norms_history.keys():
        grad_norms_history[name].append(snapshot.layer_norms.get(name, 0.0))
        ema_baselines_history[name].append(snapshot.ema_baselines.get(name, 0.0))
    
    spike_flags.append(snapshot.spike_detected)
    
    optimizer.step()

print("✅ Data collection complete!")

In [ ]:
# Plot gradient norms for the most interesting layer
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Gradient norms with EMA baseline
layer_name = 'fc2.weight'
steps = range(len(grad_norms_history[layer_name]))

axes[0].plot(steps, grad_norms_history[layer_name], 
             label='Gradient Norm', linewidth=2, color='#2563eb')
axes[0].plot(steps, ema_baselines_history[layer_name], 
             label='EMA Baseline', linewidth=2, linestyle='--', color='#16a34a')

# Highlight spike detection
spike_steps = [i for i, flag in enumerate(spike_flags) if flag]
for spike_step in spike_steps:
    axes[0].axvline(x=spike_step, color='#dc2626', linestyle=':', alpha=0.7, linewidth=2)
    axes[0].scatter([spike_step], [grad_norms_history[layer_name][spike_step]], 
                   color='#dc2626', s=100, zorder=5, marker='X')

axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Gradient Norm', fontsize=12)
axes[0].set_title(f'Gradient Monitoring: {layer_name}', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Plot 2: Spike ratio (norm / baseline)
spike_ratios = [
    norm / baseline if baseline > 0 else 0 
    for norm, baseline in zip(grad_norms_history[layer_name], ema_baselines_history[layer_name])
]

axes[1].plot(steps, spike_ratios, linewidth=2, color='#7c3aed')
axes[1].axhline(y=10.0, color='#dc2626', linestyle='--', linewidth=2, label='Threshold (10x)')

for spike_step in spike_steps:
    axes[1].axvline(x=spike_step, color='#dc2626', linestyle=':', alpha=0.7, linewidth=2)
    axes[1].scatter([spike_step], [spike_ratios[spike_step]], 
                   color='#dc2626', s=100, zorder=5, marker='X', label='Spike Detected' if spike_step == spike_steps[0] else '')

axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('Spike Ratio (norm / baseline)', fontsize=12)
axes[1].set_title('Spike Detection Ratio', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('gradient_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Visualization saved as 'gradient_monitoring.png'")
print(f"   Spikes detected at steps: {spike_steps}")

## 🎭 Compare All Three Actions

StabilityGuard supports three remediation strategies:
1. **skip**: Skip the optimizer update (safest)
2. **rollback**: Restore previous model checkpoint
3. **raise**: Throw an exception for debugging

In [ ]:
def train_with_action(action_type, n_steps=20, inject_at=10):
    """Train model with specified action type."""
    model = SimpleNet().to(device)
    base_optimizer = optim.Adam(model.parameters(), lr=0.001)
    optimizer = GuardedOptimizer(
        optimizer=base_optimizer,
        model=model,
        action=action_type,
        threshold=10.0,
        warmup_steps=5
    )
    
    losses = []
    spike_detected = False
    
    try:
        for step in range(n_steps):
            optimizer.zero_grad()
            outputs = model(X_train)
            loss = criterion(outputs, y_train)
            loss.backward()
            
            if step == inject_at:
                inject_spike(model, magnitude=500.0)
            
            optimizer.step()
            losses.append(loss.item())
            
            if hasattr(optimizer, '_last_snapshot') and optimizer._last_snapshot:
                if optimizer._last_snapshot.spike_detected:
                    spike_detected = True
                    
    except Exception as e:
        print(f"   ⚠️  Exception raised: {type(e).__name__}")
        spike_detected = True
    
    return losses, spike_detected

# Compare all three actions
print("Comparing remediation strategies:\n")

results = {}
for action in ['skip', 'rollback', 'raise']:
    print(f"Testing '{action}' action...")
    losses, detected = train_with_action(action)
    results[action] = losses
    status = "✅ Spike handled" if detected else "❌ No spike"
    print(f"   {status}\n")

In [ ]:
# Visualize comparison
plt.figure(figsize=(12, 6))

colors = {'skip': '#2563eb', 'rollback': '#16a34a', 'raise': '#dc2626'}
for action, losses in results.items():
    if losses:  # Only plot if we have data (raise might have stopped early)
        plt.plot(losses, label=f'{action.capitalize()} Action', 
                linewidth=2, color=colors[action], marker='o', markersize=4)

plt.axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='Spike Injection')
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Comparison of Remediation Strategies', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('action_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Comparison saved as 'action_comparison.png'")

## 🔍 Inspect Spike Reports

StabilityGuard generates detailed spike reports for forensic analysis:

In [ ]:
# Train and capture a spike report
model = SimpleNet().to(device)
base_optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = GuardedOptimizer(
    optimizer=base_optimizer,
    model=model,
    action='skip',
    threshold=10.0,
    warmup_steps=5
)

# Train until we capture a spike
for step in range(15):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    
    if step == 10:
        inject_spike(model, magnitude=500.0)
    
    optimizer.step()
    
    # Check for spike report
    if hasattr(optimizer, '_last_snapshot') and optimizer._last_snapshot:
        snapshot = optimizer._last_snapshot
        if snapshot.spike_detected:
            print("\n🔍 SPIKE REPORT")
            print("=" * 60)
            print(f"Step: {snapshot.step}")
            print(f"Worst Layer: {snapshot.worst_layer}")
            print(f"Worst Norm: {snapshot.worst_norm:.2f}")
            print(f"Baseline: {snapshot.worst_baseline:.2f}")
            print(f"Spike Ratio: {snapshot.worst_norm / snapshot.worst_baseline:.2f}x")
            print(f"\nTop 5 Layers by Gradient Norm:")
            sorted_layers = sorted(snapshot.layer_norms.items(), 
                                 key=lambda x: x[1], reverse=True)[:5]
            for name, norm in sorted_layers:
                baseline = snapshot.ema_baselines.get(name, 0.0)
                ratio = norm / baseline if baseline > 0 else 0
                print(f"  {name:20s}: {norm:8.2f} (baseline: {baseline:6.2f}, ratio: {ratio:5.1f}x)")
            print("=" * 60)
            break

## 🎓 Key Takeaways

1. **One-line integration**: Just wrap your optimizer with `GuardedOptimizer`
2. **Automatic detection**: EMA-based baseline tracking catches spikes automatically
3. **Three strategies**: Choose skip, rollback, or raise based on your needs
4. **Zero overhead**: Minimal performance impact during normal training
5. **Detailed logging**: Comprehensive spike reports for debugging

## 🚀 Next Steps

- Try StabilityGuard on your own models
- Experiment with different thresholds and actions
- Integrate with W&B or MLflow for experiment tracking
- Check out the [full documentation](https://github.com/ashwinmsad1/stabilityguard)

## 📚 Additional Resources

- [GitHub Repository](https://github.com/ashwinmsad1/stabilityguard)
- [GPT-2 Pretraining Example](gpt2_pretrain.py)
- [LLaMA Fine-tuning Example](llama_finetune.py)
- [API Documentation](https://github.com/ashwinmsad1/stabilityguard#api-reference)